# 01. Telemetry quality EDA

Полная неразрушающая диагностика двух телеметрий: timestamp, missing/inf/non-numeric, distributions, constant and near-constant signals, frozen intervals, abrupt jumps and noise indicators.

Аномалии **не удаляются**. Observed min/max не интерпретируются как технологические limits. Robust jump rule — диагностический флаг, а не доказанная авария.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display, Markdown

HERE = Path.cwd().resolve()
EDA_DIR = HERE if HERE.name == 'eda' else HERE / 'eda'
DATA_DIR = EDA_DIR.parent / 'data'
ARTIFACTS = EDA_DIR / 'artifacts'
ARTIFACTS.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(EDA_DIR))
from eda_utils import load_telemetry, telemetry_summary, timestamp_audit, select_variable_tags
pd.set_option('display.max_columns', 100)
FILES = {'AVT': DATA_DIR / 'avt_tags.csv', '24-2000': DATA_DIR / '242000_tags.csv'}
data = {name: load_telemetry(path) for name, path in FILES.items()}
{name: df.shape for name, df in data.items()}

## Timestamp integrity and 10-minute grid

In [ ]:
time_audit = pd.DataFrame([timestamp_audit(df, name) for name, df in data.items()])
display(time_audit)
time_audit.to_csv(ARTIFACTS / 'telemetry_timestamp_audit.csv', index=False)
fig = px.bar(time_audit, x='source', y=['bad_timestamps', 'duplicated_timestamps', 'missing_expected_10min_points'],
             barmode='group', title='Timestamp issues')
fig.show()

## Per-tag audit

`longest_constant_interval_*` считает последовательные повторы; `abrupt_jump_count_robust` использует median(|Δx|) + 10·MAD(|Δx|). Для дискретных/режимных тегов флаги нужно трактовать особо осторожно.

In [ ]:
summaries = []
for name, df in data.items():
    s = telemetry_summary(df, name)
    s.to_csv(ARTIFACTS / f'{name.lower().replace("-", "_")}_tag_quality.csv', index=False)
    summaries.append(s)
quality = pd.concat(summaries, ignore_index=True)
display(quality)
quality.to_csv(ARTIFACTS / 'all_telemetry_tag_quality.csv', index=False)

## Missingness, constants, frozen intervals and jumps

In [ ]:
for metric, title in [
    ('missing_pct', 'Missing values, %'),
    ('longest_constant_interval_hours', 'Longest repeated-value interval, hours'),
    ('abrupt_jump_count_robust', 'Robust abrupt-jump flags'),
    ('noise_ratio_diff_std_to_signal_std', 'Noise proxy: std(|Δx|) / std(x)'),
]:
    top = quality.sort_values(metric, ascending=False).head(30)
    px.bar(top, x=metric, y='tag', color='source', orientation='h', height=700, title=title).show()

flags = quality.query('is_constant or is_near_constant or longest_constant_interval_hours >= 6 or abrupt_jump_count_robust > 0')
display(flags.sort_values(['is_constant', 'longest_constant_interval_hours'], ascending=False))
flags.to_csv(ARTIFACTS / 'telemetry_diagnostic_flags.csv', index=False)

## Distribution gallery

Для читаемости в каждом источнике автоматически выбираются наиболее динамичные теги. Полная статистика всех тегов есть в таблице выше.

In [ ]:
selected = {}
for name, df in data.items():
    tags = select_variable_tags(df, n=8)
    selected[name] = tags
    sample = df[['date', *tags]].iloc[::12].melt(id_vars='date', var_name='tag', value_name='value')
    px.histogram(sample, x='value', facet_col='tag', facet_col_wrap=2, nbins=80, height=900,
                 title=f'{name}: distributions of dynamic tags (subsampled only for rendering)').update_yaxes(matches=None).show()
selected

## Time-series overview and observed ranges

In [ ]:
for name, df in data.items():
    tags = selected[name]
    long = df[['date', *tags]].set_index('date').resample('6h').median().reset_index().melt('date', var_name='tag', value_name='value')
    px.line(long, x='date', y='value', facet_row='tag', height=180 * len(tags), title=f'{name}: 6-hour median trends').update_yaxes(matches=None).show()

observed = quality[['source','tag','min','p01','median','p99','max']].copy()
observed['warning'] = 'Observed range only; NOT an operating or safety limit'
display(observed)
observed.to_csv(ARTIFACTS / 'observed_ranges_not_limits.csv', index=False)

## Interpretation guardrails

- Константа может быть нормальным дискретным состоянием; frozen sensor подтверждается только со справочником тегов.
- Скачок может быть реальным режимным переходом.
- Высокая шумность — приоритет для проверки, а не автоматическое основание удалить тег.
- До моделирования нужны временные splits; random split запрещён.